# BODAQS Event Schema Test Harness - Self-scoped

This developer notebook preprocesses one input session into one configured library. It is intended for event-schema tuning, not routine library browsing.

In [1]:
from pathlib import Path
import sys

from IPython.display import display


def find_analysis_dir(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "bodaqs_analysis").is_dir():
            return candidate
        analysis = candidate / "analysis"
        if (analysis / "bodaqs_analysis").is_dir():
            return analysis
    raise RuntimeError("Could not find the BODAQS analysis package root from the current working directory.")


ANALYSIS_DIR = find_analysis_dir()
if str(ANALYSIS_DIR) not in sys.path:
    sys.path.insert(0, str(ANALYSIS_DIR))

LIBRARIES_ROOT = Path.home() / "OneDrive" / "BODAQS-data"
LIBRARY_ID = "archie"

from bodaqs_analysis.library_api import LibraryAdapter
from bodaqs_analysis.ui import make_preprocess_runtime_settings_editor

adapter = LibraryAdapter(LIBRARIES_ROOT)
libraries = {item["library_id"]: item for item in adapter.list_libraries()}
if LIBRARY_ID not in libraries:
    available = ", ".join(sorted(libraries)) or "none found"
    raise ValueError(f"Library {LIBRARY_ID!r} was not found. Available libraries: {available}")

LIBRARY_ROOT = Path(libraries[LIBRARY_ID]["root"])

runtime_settings_editor = make_preprocess_runtime_settings_editor(
    artifacts_dir=LIBRARY_ROOT,
    preprocess_profile_path=ANALYSIS_DIR / "config" / "preprocess_profiles" / "suspension_default_v1.json",
    bike_profile_path=ANALYSIS_DIR / "config" / "bike_profiles" / "example_enduro_bike_v1.json",
    generic_log_metadata_paths=[ANALYSIS_DIR / "config" / "log_metadata_examples"],
    fit_dir=Path.home() / "OneDrive" / "BODAQS-data" / "sources" / "ben-stevo-local" / "fit",
    fit_bindings_path=ANALYSIS_DIR / "config" / "fit_bindings_v1.json",
    prompt_for_descriptions=False,
    show_log_dir=False,
    show_prompt_for_descriptions=False,
    show_run_tz_label=False,
)

print(f"Analysis package root: {ANALYSIS_DIR}")
print(f"Library root: {LIBRARY_ROOT}")
display(runtime_settings_editor.ui)


Analysis package root: C:\Users\benco\dev\BODAQS\analysis
Library root: C:\Users\benco\OneDrive\BODAQS-data\archie


In [2]:
# INPUT_PATH may be a logger .zip bundle or a legacy loose .CSV file.
INPUT_PATH = "C:/Users/benco/OneDrive/BODAQS-private/event schema tuning/260613_103335.zip"
CSV_PATH = INPUT_PATH

# Set to None to use the schema path from the preprocess profile.
# SCHEMA_PATH_OVERRIDE = ANALYSIS_DIR / "event schema" / "event_schema - Basic.yaml"
SCHEMA_PATH_OVERRIDE = None

# Optional same-stem override. Leave as None for same-stem discovery, then generic fallbacks.
LOG_METADATA_PATH = None

RUN_TZ_LABEL = "AWST"
LOG_LEVEL = "INFO"


In [3]:
import logging

import pandas as pd

from bodaqs_analysis.artifacts import (
    ArtifactStore,
    copy_raw_csv_to_source,
    copy_session_aux_sources,
    ensure_run_is_new,
    ensure_session_is_new,
    make_run_id,
    save_session_artifacts,
    write_events_partitioned_by_schema_id,
    write_metrics_partitioned_by_schema_id,
    write_run_manifest,
    write_session_manifest,
)
from bodaqs_analysis.pipeline import preprocess_session
from bodaqs_analysis.preprocess_profile import load_preprocess_config
from bodaqs_analysis.session_archive import prepare_session_input


def _optional_path(value):
    if value is None:
        return None
    text = str(value).strip()
    return Path(text) if text else None


def _path_for_manifest(value):
    return str(value) if value is not None else None


runtime_settings = runtime_settings_editor.get_settings()
ARTIFACTS_DIR = Path(runtime_settings["artifacts_dir"])
PREPROCESS_PROFILE_PATH = Path(runtime_settings["preprocess_profile_path"])
BIKE_PROFILE_PATH = runtime_settings.get("bike_profile_path")
GENERIC_LOG_METADATA_PATHS = runtime_settings.get("generic_log_metadata_paths") or []
FIT_DIR = runtime_settings.get("fit_dir")
FIT_BINDINGS_PATH = runtime_settings.get("fit_bindings_path")
LOGGER_TIMEZONE = runtime_settings.get("logger_timezone")

input_path = Path(INPUT_PATH)
schema_override = _optional_path(SCHEMA_PATH_OVERRIDE)
log_metadata_path = _optional_path(LOG_METADATA_PATH)

if ARTIFACTS_DIR.resolve() != LIBRARY_ROOT.resolve():
    print(
        "Warning: ARTIFACTS_DIR differs from the configured library root; "
        "the harness will write to ARTIFACTS_DIR as a scratch artifact target.\n"
        f"  LIBRARY_ROOT: {LIBRARY_ROOT}\n"
        f"  ARTIFACTS_DIR: {ARTIFACTS_DIR}"
    )
if not input_path.exists():
    raise FileNotFoundError(f"INPUT_PATH does not exist: {input_path}")
if not PREPROCESS_PROFILE_PATH.exists():
    raise FileNotFoundError(f"Preprocess profile does not exist: {PREPROCESS_PROFILE_PATH}")
if BIKE_PROFILE_PATH is None:
    raise ValueError("Bike profile path is blank; this harness expects bike-profile normalization/transforms.")
if not Path(BIKE_PROFILE_PATH).exists():
    raise FileNotFoundError(f"Bike profile does not exist: {BIKE_PROFILE_PATH}")

preprocess_config = load_preprocess_config(PREPROCESS_PROFILE_PATH)
schema_path = schema_override or Path(str(preprocess_config.get("schema_path") or ""))
if not schema_path.exists():
    raise FileNotFoundError(f"Event schema does not exist: {schema_path}")

fit_import = dict(preprocess_config.get("fit_import") or {})
if fit_import.get("enabled"):
    fit_import["fit_dir"] = str(FIT_DIR) if FIT_DIR is not None else None
    fit_import["bindings_path"] = str(FIT_BINDINGS_PATH) if FIT_BINDINGS_PATH is not None else None

level = getattr(logging, str(LOG_LEVEL or "INFO").upper(), logging.INFO)
root = logging.getLogger()
root.setLevel(level)
for handler in root.handlers:
    handler.setLevel(level)

store = ArtifactStore(ARTIFACTS_DIR)
run_id_base = make_run_id(tz_label=RUN_TZ_LABEL)
run_id = run_id_base
suffix = 1
while store.run_dir(run_id).exists():
    run_id = f"{run_id_base}_{suffix:02d}"
    suffix += 1
ensure_run_is_new(store, run_id=run_id, force=False)

with prepare_session_input(input_path) as session_input:
    resolved_log_metadata_path = log_metadata_path or session_input.log_metadata_path
    results = preprocess_session(
        str(session_input.csv_path),
        str(schema_path),
        preprocess_profile_path=PREPROCESS_PROFILE_PATH,
        log_metadata_path=str(resolved_log_metadata_path) if resolved_log_metadata_path is not None else None,
        generic_log_metadata_paths=None if resolved_log_metadata_path is not None else GENERIC_LOG_METADATA_PATHS,
        bike_profile_path=BIKE_PROFILE_PATH,
        fit_import=fit_import,
        timezone=LOGGER_TIMEZONE,
        strict=bool(preprocess_config.get("strict", True)),
    )

    session = results["session"]
    session_id = str(session["session_id"])
    schema = results.get("schema")
    events_df = results.get("events", pd.DataFrame())
    metrics_df = results.get("metrics", pd.DataFrame())

    ensure_session_is_new(store, run_id=run_id, session_id=session_id, force=False)

    source_sha256 = copy_raw_csv_to_source(
        store=store,
        run_id=run_id,
        session_id=session_id,
        csv_path=session_input.csv_path,
    )
    source_manifest = session_input.source_manifest(
        source_path="source/input.csv",
        source_sha256=source_sha256,
    )

aux_sources = copy_session_aux_sources(
    store=store,
    run_id=run_id,
    session_id=session_id,
    aux_sources=session.get("source", {}).get("aux_sources"),
)

save_session_artifacts(
    store,
    run_id=run_id,
    session_id=session_id,
    session_df=session["df"],
    session_meta=session["meta"],
    secondary_stream_dfs=session.get("stream_dfs"),
    secondary_stream_meta=session.get("meta", {}).get("secondary_streams"),
)

write_session_manifest(
    store,
    run_id=run_id,
    session_id=session_id,
    contracts={"session": "v0.x", "events": "v0.x", "metrics": "v0.x"},
    source=source_manifest,
    aux_sources=aux_sources,
    summary={
        "n_rows": int(len(session["df"])),
        "n_events": int(len(events_df)) if isinstance(events_df, pd.DataFrame) else 0,
        "n_metrics": int(len(metrics_df)) if isinstance(metrics_df, pd.DataFrame) else 0,
    },
)

if isinstance(events_df, pd.DataFrame) and not events_df.empty:
    write_events_partitioned_by_schema_id(
        store=store,
        run_id=run_id,
        session_id=session_id,
        events_df=events_df,
        schema_path=schema_path,
    )

if isinstance(metrics_df, pd.DataFrame) and not metrics_df.empty:
    write_metrics_partitioned_by_schema_id(
        store=store,
        run_id=run_id,
        session_id=session_id,
        metrics_df=metrics_df,
    )

write_run_manifest(
    store,
    run_id=run_id,
    session_ids=[session_id],
    timezone_label=RUN_TZ_LABEL,
    pipeline_config={
        "purpose": "event_schema_test_harness",
        "input_path": str(input_path),
        "schema_path": str(schema_path),
        "schema_source": "override" if schema_override is not None else "preprocess_profile",
        "preprocess_profile_path": str(PREPROCESS_PROFILE_PATH),
        "bike_profile_path": _path_for_manifest(BIKE_PROFILE_PATH),
        "log_metadata_path": _path_for_manifest(log_metadata_path),
        "generic_log_metadata_paths": [str(p) for p in GENERIC_LOG_METADATA_PATHS],
        "logger_timezone_fallback": LOGGER_TIMEZONE,
        "fit_dir": _path_for_manifest(FIT_DIR),
        "fit_bindings_path": _path_for_manifest(FIT_BINDINGS_PATH),
        "strict": bool(preprocess_config.get("strict", True)),
        "n_inputs": 1,
    },
)

session_key = f"{run_id}::{session_id}"
artifact_session_dir = store.session_dir(run_id, session_id)
source_meta = session.get("source", {}) if isinstance(session.get("source"), dict) else {}
signals = session.get("meta", {}).get("signals", {}) if isinstance(session.get("meta"), dict) else {}
primary_signals = [
    col
    for col, info in signals.items()
    if isinstance(info, dict) and info.get("processing_role") == "primary_analysis"
]

print(f"session_id: {session_id}")
print(f"run_id: {run_id}")
print(f"artifact_session_dir: {artifact_session_dir}")
print(f"schema_path: {schema_path}")
print(f"events: {len(events_df) if isinstance(events_df, pd.DataFrame) else 0}")
print(f"metrics: {len(metrics_df) if isinstance(metrics_df, pd.DataFrame) else 0}")
print("primary analysis signals:")
for col in sorted(primary_signals):
    print(f"  {col}")


  LIBRARY_ROOT: C:\Users\benco\OneDrive\BODAQS-data\archie
  ARTIFACTS_DIR: C:\Users\benco\OneDrive\BODAQS-private\event schema tuning\artifacts


rebounds_all(rear): trigger 'rear_wheel_vel_dom_wheel [mm/s]' is flat (min=max=0)
compressions_all(rear): trigger 'rear_wheel_vel_dom_wheel [mm/s]' is flat (min=max=0)


session_id: 260613_103335
run_id: run_2026-06-16T21-13-55_AWST
artifact_session_dir: C:\Users\benco\OneDrive\BODAQS-private\event schema tuning\artifacts\runs\run_2026-06-16T21-13-55_AWST\sessions\260613_103335
schema_path: C:\Users\benco\OneDrive\BODAQS-private\event schema tuning\schema\event_schema_phased.yaml
events: 3035
metrics: 3035
primary analysis signals:
  front_wheel_acc_dom_wheel [mm/s^2]
  front_wheel_disp_dom_wheel [1]_op_zeroed_norm
  front_wheel_disp_dom_wheel [mm]
  front_wheel_vel_dom_wheel [mm/s]
  rear_wheel_acc_dom_wheel [mm/s^2]
  rear_wheel_disp_dom_wheel [1]_op_zeroed_norm
  rear_wheel_disp_dom_wheel [mm]
  rear_wheel_vel_dom_wheel [mm/s]


In [4]:
from bodaqs_analysis.widgets.event_browser import make_event_browser_widget_for_loader
from bodaqs_analysis.widgets.loaders import (
    load_all_events_for_selected,
    load_all_metrics_for_selected,
    make_session_loader,
)

if "store" not in globals() or "session_key" not in globals():
    raise RuntimeError("Run the preprocessing cell first.")

key_to_ref = {session_key: (run_id, session_id)}
session_loader = make_session_loader(store=store, key_to_ref=key_to_ref)
events_df_sel = load_all_events_for_selected(store, key_to_ref=key_to_ref)
metrics_df_sel = load_all_metrics_for_selected(store, key_to_ref=key_to_ref)

if events_df_sel.empty:
    raise ValueError("No events were written for this session.")

browser = make_event_browser_widget_for_loader(
    schema,
    events_df_sel,
    session_loader=session_loader,
    metrics_df=metrics_df_sel if not metrics_df_sel.empty else None,
    session_key_col="session_key",
)

display(browser["ui"])
